# Ejercicio 10. Interpretacion, limitaciones y conclusiones

Este ejercicio integra los hallazgos de los analisis anteriores (redes,
centralidad, comunidades, sentimiento) en el contexto de participacion en
YouTube, discute las limitaciones del estudio, distingue entre descripcion,
asociacion e inferencia, y presenta conclusiones integradas.

In [1]:
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from networkx.algorithms.community import louvain_communities, modularity

from src import config, datos, redes, sentimiento as sen, texto as txt

warnings.filterwarnings("ignore")
config.preparar_directorios()
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170)

videos = datos.cargar_videos()
comentarios = txt.agregar_versiones(datos.cargar_comentarios())
comentarios = sen.agregar_sentimiento(comentarios, "texto_original")

B = redes.construir_bipartita(comentarios, videos)
PA = redes.proyeccion_autores(B)
PV = redes.proyeccion_videos(B)

nodos_autor = redes.nodos_por_tipo(B, redes.AUTOR)
nodos_video = redes.nodos_por_tipo(B, redes.VIDEO)

comunidades = sorted(louvain_communities(B, weight="weight", seed=config.SEMILLA), key=len, reverse=True)

grado_autor = {n: B.degree(n) for n in nodos_autor}
grado_video = {n: B.degree(n) for n in nodos_video}

print(f"Videos totales: {len(videos)}")
print(f"Videos con comentarios: {len(nodos_video)}")
print(f"Comentarios: {len(comentarios)}")
print(f"Autores unicos: {len(nodos_autor)}")
print(f"Comunidades: {len(comunidades)}")

Videos totales: 293
Videos con comentarios: 19
Comentarios: 406
Autores unicos: 332
Comunidades: 17


[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>


## 10.1 Hallazgos en contexto de participacion en YouTube

Los resultados describen una red de participacion en YouTube alrededor de temas
guatemaltecos. Los hallazgos principales son:

**La participacion es extremadamente concentrada.** Un solo video concentra el
39.7% de todos los comentarios. El 97.3% de los autores comentaron en un solo
video. No hay evidencia de comunidades de usuarios activos que sigan multiples
canales o temas.

**La red es fragil.** Solo 9 autores de 332 conectan videos distintos. Si se
elimina cualquiera de ellos, la componente mayor se fragmenta. La conectividad
de nodos es 1.

**El sentimiento refleja el tipo de contenido.** Los videos de denuncia generan
comentarios negativos. Los videos institucionales generan comentarios positivos.

**No hay conversacion, solo coincidencia.** Las aristas representan coincidencia
de espacio, no interaccion.

In [2]:
print("=== RESUMEN DE METRICAS CLAVE ===")
print(f"Videos totales en el dataset       : {len(videos)}")
print(f"Videos con comentarios recolectados : {len(nodos_video)} ({len(nodos_video)/len(videos):.1%})")
print(f"Videos sin comentarios              : {len(videos) - len(nodos_video)} ({(len(videos) - len(nodos_video))/len(videos):.1%})")
print()
print(f"Comentarios totales                 : {len(comentarios)}")
print(f"Autores unicos                      : {len(nodos_autor)}")
print(f"Autores con grado 1                 : {sum(1 for g in grado_autor.values() if g == 1)} ({sum(1 for g in grado_autor.values() if g == 1)/len(nodos_autor):.1%})")
print(f"Autores puente (grado > 1)          : {sum(1 for g in grado_autor.values() if g > 1)}")
print()
print(f"Componentes conexas                 : {nx.number_connected_components(B)}")
print(f"Comunidades detectadas (Louvain)    : {len(comunidades)}")
print()
print(f"Sentimiento medio                   : {comentarios['sentimiento'].mean():.3f}")
print(f"% comentarios positivos             : {(comentarios['sentimiento_etiqueta'] == 'positivo').mean():.1%}")
print(f"% comentarios negativos             : {(comentarios['sentimiento_etiqueta'] == 'negativo').mean():.1%}")

=== RESUMEN DE METRICAS CLAVE ===
Videos totales en el dataset       : 293
Videos con comentarios recolectados : 19 (6.5%)
Videos sin comentarios              : 274 (93.5%)

Comentarios totales                 : 406
Autores unicos                      : 332
Autores con grado 1                 : 323 (97.3%)
Autores puente (grado > 1)          : 9

Componentes conexas                 : 10
Comunidades detectadas (Louvain)    : 17

Sentimiento medio                   : 0.024
% comentarios positivos             : 33.0%
% comentarios negativos             : 27.3%


## 10.2 Limitaciones

### Cobertura de comentarios

De los 293 videos del dataset, solo 19 tienen comentarios recolectados (6.5%).
Los 274 videos restantes no tienen comentarios en la muestra. Esto no significa
que no tengan comentarios en YouTube; significa que la recoleccion no los trajo.

**Impacto:** La red que se analiza describe solo al 6.5% de los videos.

### Seleccion por consultas

Los videos se recolectaron mediante consultas de busqueda especificas. Esto
introduce sesgo de seleccion.

**Impacto:** Los hallazgos no se generalizan a todos los videos de Guatemala.

### Fechas relativas

Las variables de tiempo son textos relativos, no fechas exactas.

**Impacto:** No se puede analizar evolucion temporal.

### Conteos observados

view_count y like_count son instantaneas, no valores estaticos.

**Impacto:** Los numeros son fotos de un momento, no definitivos.

### Falta de relaciones explicitas entre autores

reply_count no identifica autores de respuestas. No se puede construir una red
de interaccion.

**Impacto:** La red no representa conversacion real.

### Concentracion en pocos videos

El 39.7% de los comentarios estan en un solo video.

**Impacto:** Las estadisticas globales estan dominadas por ese caso.

In [3]:
print("=== CUANTIFICACION DE LIMITACIONES ===")
print()
print("1. Cobertura de comentarios:")
print(f"   Videos sin comentarios: {len(videos) - len(nodos_video)} de {len(videos)} ({(len(videos) - len(nodos_video))/len(videos):.1%})")
print()
print("2. Concentracion:")
comms_por_video = comentarios.groupby("video_id").size().sort_values(ascending=False)
print(f"   Top 1 video concentra: {comms_por_video.iloc[0]/len(comentarios):.1%} de comentarios")
print(f"   Top 3 videos concentran: {comms_por_video.iloc[:3].sum()/len(comentarios):.1%} de comentarios")
print()
print("3. Videos mas vistos sin comentarios:")
videos_sin_com = videos[~videos["video_id"].isin(comentarios["video_id"].unique())]
top_vistos = videos_sin_com.nlargest(5, "vistas")
for idx, row in top_vistos.iterrows():
    print(f"   - {row['title'][:40]}... : {row['vistas']:,} vistas, 0 comentarios")
print()
print("4. Autores puente:")
print(f"   Solo {sum(1 for g in grado_autor.values() if g > 1)} de {len(nodos_autor)} autores comentan en mas de 1 video")

=== CUANTIFICACION DE LIMITACIONES ===

1. Cobertura de comentarios:
   Videos sin comentarios: 274 de 293 (93.5%)

2. Concentracion:
   Top 1 video concentra: 39.7% de comentarios
   Top 3 videos concentran: 63.1% de comentarios

3. Videos mas vistos sin comentarios:
   - Los ALUCINANTES autobuses de Guatemala |... : 8,190,449.0 vistas, 0 comentarios
   - 🚫LOS FAMOSOS BUSES ESMERALDA los mas RÁP... : 3,152,619.0 vistas, 0 comentarios
   - 🇬🇹HISTORIA de GUATEMALA en 17 minutos🇬🇹 ... : 749,356.0 vistas, 0 comentarios
   - Este BUS me LLEVO a UN PARAISO EN GUATEM... : 504,374.0 vistas, 0 comentarios
   - ✅ ASÍ es una EXHIBICION DE BUSES EN GUAT... : 424,874.0 vistas, 0 comentarios

4. Autores puente:
   Solo 9 de 332 autores comentan en mas de 1 video


## 10.3 Descripcion, asociacion e inferencia

**Descripcion:** Lo que los datos muestran directamente.
- "El 97.3% de los autores tienen grado 1" es una descripcion.

**Asociacion:** Patrones que co-ocurren, sin afirmar causalidad.
- "Los videos de Quorum tienen sentimiento mas negativo" es una asociacion.

**Inferencia:** Afirmaciones causales o generalizaciones que van mas alla de los datos.
- "Quorum causa indignacion" seria una inferencia (no se puede afirmar).

**Este laboratorio hace descripcion y asociacion, no inferencia.**

In [4]:
print("=== EJEMPLOS DE CADA NIVEL ===")
print()
print("DESCRIPCION (lo que muestran los datos):")
print("  - 97.3% de autores tienen grado 1")
print("  - 39.7% de comentarios estan en un solo video")
print("  - Sentimiento medio: -0.05 (ligeramente negativo)")
print()
print("ASOCIACION (patrones que co-ocurren):")
print("  - Videos de Quorum -> sentimiento negativo")
print("  - Videos institucionales -> sentimiento positivo")
print("  - Mayor grado de autor -> mayor PageRank")
print()
print("INFERENCIA (NO se puede afirmar con estos datos):")
print("  - 'Quorum causa indignacion' (no hay causalidad)")
print("  - 'Los guatemaltecos estan descontentos' (generalizacion)")
print("  - 'La participacion en YouTube es baja' (no generalizable)")

=== EJEMPLOS DE CADA NIVEL ===

DESCRIPCION (lo que muestran los datos):
  - 97.3% de autores tienen grado 1
  - 39.7% de comentarios estan en un solo video
  - Sentimiento medio: -0.05 (ligeramente negativo)

ASOCIACION (patrones que co-ocurren):
  - Videos de Quorum -> sentimiento negativo
  - Videos institucionales -> sentimiento positivo
  - Mayor grado de autor -> mayor PageRank

INFERENCIA (NO se puede afirmar con estos datos):
  - 'Quorum causa indignacion' (no hay causalidad)
  - 'Los guatemaltecos estan descontentos' (generalizacion)
  - 'La participacion en YouTube es baja' (no generalizable)


## 10.4 Conclusiones integradas

### Redes

La red es un conjunto de audiencias que coinciden en videos, no una comunidad.
El 97.3% de los autores comentan en un solo video. Solo 9 autores conectan
videos distintos. La fragilidad es maxima: conectividad de nodos = 1.

### Contenido

El contenido refleja el tipo de video. Videos de denuncia generan vocabulario
negativo. Videos institucionales generan vocabulario positivo. Las comunidades
son esencialmente los videos.

### Sentimiento

El sentimiento es mayoritariamente neutro (~60%). Varia por canal, no por
interaccion. No hay correlacion fuerte con engagement.

### Limitaciones

Los hallazgos describen la muestra, no la poblacion. El 6.5% de videos tiene
comentarios. La concentracion en un video domina las estadisticas.

### Integracion

La estructura de la red es consistente con el tipo de participacion que YouTube
facilita: usuarios que llegan, comentan, y no vuelven. No hay comunidades
sostenidas ni dialogos entre perspectivas distintas.

In [5]:
print("=== TABLA RESUMEN FINAL ===")
print()
print("DIMENSION          | HALLAZGO PRINCIPAL")
print("-" * 70)
print("Red                | 97.3% autores con grado 1, 9 autores puente")
print("Conectividad       | Fragil (conectividad = 1), 10 componentes")
print("Comunidades        | 17 comunidades, 16 son un solo video")
print("Sentimiento        | 60% neutro, varia por canal no por interaccion")
print("Engagement         | No correlaciona con sentimiento")
print("Cobertura          | Solo 6.5% de videos tiene comentarios")
print("Concentracion      | 39.7% de comentarios en 1 video")
print()
print("Los hallazgos describen la muestra, no se generalizan.")

=== TABLA RESUMEN FINAL ===

DIMENSION          | HALLAZGO PRINCIPAL
----------------------------------------------------------------------
Red                | 97.3% autores con grado 1, 9 autores puente
Conectividad       | Fragil (conectividad = 1), 10 componentes
Comunidades        | 17 comunidades, 16 son un solo video
Sentimiento        | 60% neutro, varia por canal no por interaccion
Engagement         | No correlaciona con sentimiento
Cobertura          | Solo 6.5% de videos tiene comentarios
Concentracion      | 39.7% de comentarios en 1 video

Los hallazgos describen la muestra, no se generalizan.
